<a href="https://colab.research.google.com/github/dengel29/colab-audio-extraction/blob/main/audioextractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install openai-whisper moviepy pydub
!apt install ffmpeg
import os
import whisper
import pandas as pd
from moviepy.editor import VideoFileClip
from pydub import AudioSegment
from pydub.silence import detect_leading_silence
from datetime import datetime
from google.colab import drive

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 5.9 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=de979d6cda3e3a453395f2ceaea790cddc2c6e29af4b3cb9c4372c31ea2a4863
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':

  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)

  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)

  elif re.match('(flt)p?( \(default\))?$', token):

  elif re.m

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

# Where the videos are (your shortcut)
BASE_FOLDER = '/content/drive/MyDrive/Compressed'

# Where the audio and transcripts will go
OUTPUT_FOLDER = '/content/drive/MyDrive/Compressed Audio Files'

# Create the output folder if it doesn't exist yet
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)
    print(f"Created new folder: {OUTPUT_FOLDER}")

LOG_FILE_PATH = os.path.join(OUTPUT_FOLDER, "transcription_log.csv")

In [4]:
import torch

# 1. Load Whisper model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
model = whisper.load_model("base", device=device)

# Helper function to trim silence from both ends
def trim_edges(audio_segment, silence_threshold=-50.0, chunk_size=10):
    # Trim leading silence
    start_trim = detect_leading_silence(audio_segment, silence_threshold, chunk_size)

    # Trim trailing silence (by reversing the audio)
    end_trim = detect_leading_silence(audio_segment.reverse(), silence_threshold, chunk_size)

    duration = len(audio_segment)
    trimmed_audio = audio_segment[start_trim:duration-end_trim]
    return trimmed_audio

def process_videos_with_logging(root_path, output_path, limit=20):
    if os.path.exists(LOG_FILE_PATH):
        log_df = pd.read_csv(LOG_FILE_PATH)
    else:
        log_df = pd.DataFrame(columns=[
            "video_name", "original_path", "video_duration_sec",
            "audio_file", "audio_duration_sec", "transcription_chars",
            "status", "timestamp"
        ])

    processed_count = 0

    for root, dirs, files in os.walk(root_path):
        for file in files:
            if processed_count >= limit:
                print(f"\n✋ Reached limit. Stopping.")
                return

            if file.endswith(('.mp4', '.mov', '.mkv')):
                video_path = os.path.join(root, file)

                # Resume logic: skip if Succeeded
                if not log_df.empty and video_path in log_df['original_path'].values:
                    if log_df.loc[log_df['original_path'] == video_path, 'status'].iloc[0] == "Succeeded":
                        continue

                processed_count += 1
                print(f"[{processed_count}/{limit}] 🎬 Processing: {file}")

                # Naming Logic
                relative_path = os.path.relpath(root, root_path).replace("/", "_").replace(".", "")
                clean_name = f"{relative_path}_{file.rsplit('.', 1)[0]}" if relative_path != "" else file.rsplit('.', 1)[0]
                full_audio_path = os.path.join(output_path, f"{clean_name}.mp3")
                full_text_path = os.path.join(output_path, f"{clean_name}.txt")

                entry = {
                    "video_name": file, "original_path": video_path,
                    "video_duration_sec": 0, "audio_file": f"{clean_name}.mp3",
                    "audio_duration_sec": 0, "transcription_chars": 0,
                    "status": "Starting", "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                }

                video = None
                try:
                    # 2. Extract and Measure Original Video
                    video = VideoFileClip(video_path)
                    entry["video_duration_sec"] = round(video.duration, 2)

                    # Temporarily save for pydub to read
                    temp_wav = "temp.wav"
                    video.audio.write_audiofile(temp_wav, codec='pcm_s16le', logger=None)

                    # 3. Trim silence from edges
                    audio_segment = AudioSegment.from_file(temp_wav)
                    trimmed_audio = trim_edges(audio_segment)

                    # 4. Save trimmed audio to Drive
                    trimmed_audio.export(full_audio_path, format="mp3", bitrate="192k")
                    entry["audio_duration_sec"] = round(len(trimmed_audio) / 1000.0, 2)

                    # 5. Transcribe
                    result = model.transcribe(full_audio_path)
                    entry["transcription_chars"] = len(result["text"])

                    with open(full_text_path, "w") as f:
                        f.write(result["text"])

                    entry["status"] = "Succeeded"

                except Exception as e:
                    print(f"❌ Error: {e}")
                    entry["status"] = f"Failed: {str(e)}"

                finally:
                    if video: video.close()
                    if os.path.exists("temp.wav"): os.remove("temp.wav")

                # Save Progress
                log_df = pd.concat([log_df, pd.DataFrame([entry])], ignore_index=True)
                log_df.to_csv(LOG_FILE_PATH, index=False)

    print(f"\n✅ All done! Log updated at: {LOG_FILE_PATH}")

# RUN IT
process_videos_with_logging(BASE_FOLDER, OUTPUT_FOLDER, limit=20)

Using device: cpu


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 126MiB/s]


[1/20] 🎬 Processing: Candice_1_Hamstring curl_zh.mp4


  warnings.warn("FP16 is not supported on CPU; using FP32 instead")

  log_df = pd.concat([log_df, pd.DataFrame([entry])], ignore_index=True)



[2/20] 🎬 Processing: Candice_2_Single leg hamstring curl_zh.mp4


...

[19/20] 🎬 Processing: Candice_22_eccentric assisted pull up w band_zh.mp4


  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



[20/20] 🎬 Processing: Candice_23_single arm cable row_zh.mp4


  warnings.warn("FP16 is not supported on CPU; using FP32 instead")




✋ Reached limit. Stopping.
